In [1]:
!pip install -q unsloth
!pip install -q bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.3/359.3 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 22.6 MB/s eta 0:00:00


In [2]:
# Install lm-evaluation-harness
!pip install -q lm-eval

# Install bigcode-evaluation-harness
!pip install -q git+https://github.com/bigcode-project/bigcode-evaluation-harness.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 5.0 MB/s eta 0:00:00


In [3]:
import os
import json
import random
import time
import subprocess
import gc
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display, HTML

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

warnings.filterwarnings('ignore')

# Seeds
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# GPU check
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU!")

# Directories
DATA_DIR = "./data"
RESULTS_DIR = "./results"
for d in [DATA_DIR, RESULTS_DIR, f"{RESULTS_DIR}/code", f"{RESULTS_DIR}/math", f"{RESULTS_DIR}/theory"]:
    os.makedirs(d, exist_ok=True)

GPU: NVIDIA A100-SXM4-80GB
   Memory: 85.2 GB


In [4]:
# Model Registry

@dataclass
class ModelConfig:
    name: str
    hf_path: str
    unsloth_path: Optional[str]
    category: str
    max_seq_length: int = 4096

# CODING MODELS (Select Top 1)
CODING_MODELS = [
    ModelConfig("CodeGemma-7B-IT", "google/codegemma-7b-it",
                "unsloth/codegemma-7b-it-bnb-4bit", "code"),
    ModelConfig("Qwen2.5-Coder-7B-Instruct", "Qwen/Qwen2.5-Coder-7B-Instruct",
                "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", "code"),
    ModelConfig("DeepSeek-Coder-6.7B-Instruct", "deepseek-ai/deepseek-coder-6.7b-instruct",
                None, "code"),
    ModelConfig("CodeLlama-7B-Instruct", "codellama/CodeLlama-7b-Instruct-hf",
                "unsloth/codellama-7b-bnb-4bit", "code"),
    ModelConfig("Seed-Coder-8B-Instruct", "ByteDance-Seed/Seed-Coder-8B-Instruct",
                "unsloth/Seed-Coder-8B-Instruct-bnb-4bit", "code"),
]

# MATH MODELS (Select Top 2)
MATH_MODELS = [
    ModelConfig("DeepSeek-Math-7B-Instruct", "deepseek-ai/deepseek-math-7b-instruct",
                None, "math"),
    ModelConfig("AceMath-7B-Instruct", "nvidia/AceMath-7B-Instruct",
                None, "math"),
    ModelConfig("Qwen2.5-Math-7B-Instruct", "Qwen/Qwen2.5-Math-7B-Instruct",
                "unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit", "math"),
    ModelConfig("Llemma-7B", "EleutherAI/llemma_7b",
                None, "math"),
    ModelConfig("WizardMath-7B-V1.1", "WizardLMTeam/WizardMath-7B-V1.1",
                None, "math"),
]

# THEORY MODELS (Select Top 2)
THEORY_MODELS = [
    ModelConfig("Llama-3.1-8B-Instruct", "meta-llama/Llama-3.1-8B-Instruct",
                "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", "theory"),
    ModelConfig("Mistral-7B-Instruct-v0.3", "mistralai/Mistral-7B-Instruct-v0.3",
                "unsloth/Mistral-7B-Instruct-v0.3-bnb-4bit", "theory"),
    ModelConfig("Gemma-2-9B-Instruct", "google/gemma-2-9b-it",
                "unsloth/gemma-2-9b-it-bnb-4bit", "theory"),
    ModelConfig("Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-7B-Instruct",
                "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", "theory"),
    ModelConfig("Falcon-H1-7B-Instruct", "tiiuae/Falcon-H1-7B-Instruct",
                "unsloth/Falcon-H1-7B-Instruct", "theory"),
]

ALL_MODELS = {"code": CODING_MODELS, "math": MATH_MODELS, "theory": THEORY_MODELS}

print("Model Registry:")
for cat, models in ALL_MODELS.items():
    top_n = 1 if cat == "code" else 2
    print(f"\n{cat.upper()} ({len(models)} models) → Select Top {top_n}:")
    for m in models:
        status = "✓ Unsloth" if m.unsloth_path else "○ Standard"
        print(f"   {status} | {m.name}")

Model Registry:

CODE (5 models) → Select Top 1:
   ✓ Unsloth | CodeGemma-7B-IT
   ✓ Unsloth | Qwen2.5-Coder-7B-Instruct
   ○ Standard | DeepSeek-Coder-6.7B-Instruct
   ✓ Unsloth | CodeLlama-7B-Instruct
   ✓ Unsloth | Seed-Coder-8B-Instruct

MATH (5 models) → Select Top 2:
   ○ Standard | DeepSeek-Math-7B-Instruct
   ○ Standard | AceMath-7B-Instruct
   ✓ Unsloth | Qwen2.5-Math-7B-Instruct
   ○ Standard | Llemma-7B
   ○ Standard | WizardMath-7B-V1.1

THEORY (5 models) → Select Top 2:
   ✓ Unsloth | Llama-3.1-8B-Instruct
   ✓ Unsloth | Mistral-7B-Instruct-v0.3
   ✓ Unsloth | Gemma-2-9B-Instruct
   ✓ Unsloth | Qwen2.5-7B-Instruct
   ✓ Unsloth | Falcon-H1-7B-Instruct


In [5]:
from google.colab import auth
from google.cloud import storage

auth.authenticate_user()
print("GCP authenticated!")

PROJECT_ID = "msda-llm-finetuning"
PROCESSED_BUCKET = "msda-llm-processed-data"
!gcloud config set project {PROJECT_ID} --quiet

# Download test data
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(PROCESSED_BUCKET)

def download_from_gcs(gcs_path, local_path):
    blob = bucket.blob(gcs_path)
    blob.download_to_filename(local_path)

print("\nDownloading test splits...")
download_from_gcs("theory/theory_test.jsonl", f"{DATA_DIR}/theory_test.jsonl")
download_from_gcs("code/code_test.jsonl", f"{DATA_DIR}/code_test.jsonl")
download_from_gcs("math/math_test.jsonl", f"{DATA_DIR}/math_test.jsonl")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

theory_test_full = load_jsonl(f"{DATA_DIR}/theory_test.jsonl")
code_test_full = load_jsonl(f"{DATA_DIR}/code_test.jsonl")
math_test_full = load_jsonl(f"{DATA_DIR}/math_test.jsonl")

print(f"\nCustom Test Data:")
print(f"   Theory: {len(theory_test_full):,}")
print(f"   Code: {len(code_test_full):,}")
print(f"   Math: {len(math_test_full):,}")

# Sample for faster evaluation
SAMPLE_SIZE = 100
theory_test = random.sample(theory_test_full, min(SAMPLE_SIZE, len(theory_test_full)))
code_test = random.sample(code_test_full, min(SAMPLE_SIZE, len(code_test_full)))
math_test = random.sample(math_test_full, min(SAMPLE_SIZE, len(math_test_full)))

print(f"\nSampled for evaluation:")
print(f"   Theory: {len(theory_test)}")
print(f"   Code: {len(code_test)}")
print(f"   Math: {len(math_test)}")

GCP authenticated!
Updated property [core/project].


Custom Test Data:
   Theory: 165
   Code: 1,944
   Math: 13,811

Sampled for evaluation:
   Theory: 100
   Code: 100
   Math: 100


In [6]:
# LLM-AS-JUDGE SETUP (STRICT VERSION with GPT-4o)

import openai
from google.colab import userdata

# Set your API key - use Colab secrets
try:
    JUDGE_API_KEY = userdata.get('OPENAI_API_KEY')
    JUDGE_API_BASE = "https://api.openai.com/v1"
    JUDGE_MODEL = "gpt-4o"  # Using GPT-4o for stricter, more accurate evaluation
    print("Using API key from Colab Secrets")
except:
    JUDGE_API_KEY = "your-api-key-here"
    JUDGE_API_BASE = "https://api.openai.com/v1"
    JUDGE_MODEL = "gpt-4o"
    print("Set your API key in JUDGE_API_KEY variable")

openai_client = openai.OpenAI(api_key=JUDGE_API_KEY, base_url=JUDGE_API_BASE)


def llm_judge_score(question: str, reference: str, response: str,
                    category: str, max_retries: int = 3) -> Dict:
    """
    Use LLM-as-Judge to score a response with STRICT evaluation criteria.
    Uses GPT-4o for more accurate and demanding scoring.
    """

    if category == "code":
        criteria = """
You are a STRICT code evaluator. Be critical and demanding. Most responses should score 2-4, not 4-5.

1. **Correctness (1-5)**: Does the code correctly solve the problem? Any bugs = max score of 3.
2. **Code Quality (1-5)**: Is the code clean, readable, well-documented? Missing docstrings/comments = max 3.
3. **Completeness (1-5)**: Does it handle edge cases? Missing error handling = max 3.
4. **Efficiency (1-5)**: Is it optimally efficient? Naive solutions = max 3.

SCORING GUIDE:
- 5 = PERFECT: Production-ready, no improvements needed
- 4 = VERY GOOD: Minor issues only, high quality
- 3 = ACCEPTABLE: Works but has clear gaps or issues
- 2 = POOR: Significant problems, barely functional
- 1 = INCORRECT: Wrong approach or doesn't work
"""
    elif category == "math":
        criteria = """
You are a STRICT mathematics evaluator. Be critical and demanding. Most responses should score 2-4.

1. **Correctness (1-5)**: Is the final answer correct? Wrong answer = max score of 2.
2. **Reasoning (1-5)**: Is every step mathematically sound? Any logical gaps = max 3.
3. **Completeness (1-5)**: Are ALL steps shown? Skipping steps = max 3.
4. **Clarity (1-5)**: Is it clearly written and easy to follow?

SCORING GUIDE:
- 5 = PERFECT: Correct answer with flawless, complete reasoning
- 4 = VERY GOOD: Correct with minor presentation issues
- 3 = ACCEPTABLE: Correct answer but reasoning has gaps
- 2 = POOR: Incorrect answer but some valid reasoning shown
- 1 = INCORRECT: Wrong answer with flawed reasoning
"""
    else:  # theory
        criteria = """
You are a STRICT Data Science theory evaluator. Be critical and demanding. Most responses should score 2-4.

1. **Accuracy (1-5)**: Is EVERY claim factually correct? Any error = max score of 3.
2. **Completeness (1-5)**: Does it cover ALL key points from the reference? Missing major concepts = max 3.
3. **Clarity (1-5)**: Is it well-organized and professionally written?
4. **Depth (1-5)**: Does it show EXPERT-level understanding, not just surface knowledge?

SCORING GUIDE:
- 5 = PERFECT: Publishable quality, comprehensive, no improvements needed
- 4 = VERY GOOD: Thorough with only minor omissions
- 3 = ACCEPTABLE: Covers basics but lacks depth or has minor errors
- 2 = POOR: Superficial, missing key concepts, or has errors
- 1 = INCORRECT: Fundamentally wrong or irrelevant

IMPORTANT: Be especially critical if the response is:
- Generic or vague (not specific to the question)
- Missing key technical details from the reference
- Containing any factual errors
"""

    prompt = f"""You are an expert STRICT evaluator for {category} responses.
Compare the model's response against the reference answer and score CRITICALLY.

{criteria}

**Question/Prompt:**
{question[:1500]}

**Reference Answer (GOLD STANDARD - compare against this):**
{reference[:2000]}

**Model Response (EVALUATE THIS):**
{response[:2000]}

CRITICAL: Compare the model response AGAINST the reference.
- If model misses key points from reference, deduct points
- If model makes any errors, deduct points
- Most responses should score 2-4, NOT 4-5
- Only give 5 for truly exceptional responses

Provide your evaluation in this exact JSON format:
{{
    "scores": {{
        "criterion_1": <score 1-5>,
        "criterion_2": <score 1-5>,
        "criterion_3": <score 1-5>,
        "criterion_4": <score 1-5>
    }},
    "overall": <average score 1-5>,
    "brief_feedback": "<specific feedback on what was wrong or missing>"
}}

Only output the JSON, nothing else."""

    for attempt in range(max_retries):
        try:
            response_obj = openai_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=400,
            )
            result_text = response_obj.choices[0].message.content.strip()

            # Parse JSON
            if "```json" in result_text:
                result_text = result_text.split("```json")[1].split("```")[0]
            elif "```" in result_text:
                result_text = result_text.split("```")[1].split("```")[0]

            result = json.loads(result_text)
            return result

        except Exception as e:
            if attempt == max_retries - 1:
                return {"scores": {}, "overall": 0, "brief_feedback": f"Error: {str(e)[:50]}"}
            time.sleep(1)

    return {"scores": {}, "overall": 0, "brief_feedback": "Failed to get judge response"}


print("LLM-as-Judge configured with STRICT evaluation!")
print(f"  Judge Model: {JUDGE_MODEL}")
print("  Note: Using stricter criteria - expect lower scores than before")

Using API key from Colab Secrets
LLM-as-Judge configured with STRICT evaluation!
  Judge Model: gpt-4o
  Note: Using stricter criteria - expect lower scores than before


In [7]:
# LM-EVALUATION-HARNESS FOR MATH MODELS

import subprocess
import json
import os
import re

def find_results_json(output_path: str) -> Optional[str]:
    """
    Find results JSON file in output directory.

    lm-eval saves to:
    output_path/MODEL_NAME_SANITIZED/results_TIMESTAMP.json

    Example:
    ./results/math/DeepSeek-Math-7B-Instruct_lm_eval/deepseek-ai__deepseek-math-7b-instruct/results_2025-12-02T09-50-57.555884.json
    """
    import glob

    # Search patterns - from most specific to least
    patterns = [
        f"{output_path}/**/results_*.json",   # Recursive search for results_*.json
        f"{output_path}/**/results.json",      # Recursive search for results.json
        f"{output_path}/*/results_*.json",     # One level deep
        f"{output_path}/*/results.json",       # One level deep
        f"{output_path}/results*.json",        # Direct in output_path
    ]

    for pattern in patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            # Return the most recent one (sorted by filename)
            found = sorted(matches)[-1]
            print(f"   Found results file: {found}")
            return found

    return None


def parse_lm_eval_stdout(stdout: str) -> Dict:
    """Parse results directly from lm-eval stdout when results.json not found."""
    results = {}

    # Match lines like: |gsm8k|1|none|5|exact_match,strict-match|↑|0.4000|±|0.1633|
    # Or: |hendrycks_math|1|none| |exact_match|↑|0.1714|±|0.0457|

    lines = stdout.split('\n')
    for line in lines:
        # Look for result lines with scores
        if '|' in line and ('exact_match' in line or 'acc' in line):
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 8:
                task_name = parts[1].strip() if len(parts) > 1 else ""
                # Find the score (usually a decimal number)
                for part in parts:
                    try:
                        score = float(part)
                        if 0 <= score <= 1:  # Valid score range
                            if task_name and task_name not in ['Groups', '']:
                                results[task_name] = score
                            break
                    except ValueError:
                        continue

    return results


def run_lm_eval_harness(model_path: str, tasks: List[str], output_path: str,
                        num_fewshot: int = 5, limit: int = 10) -> Dict:
    """Run lm-evaluation-harness with fixed result parsing."""
    import glob

    os.makedirs(output_path, exist_ok=True)
    tasks_str = ",".join(tasks)

    cmd = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_path},trust_remote_code=True,load_in_4bit=True",
        "--tasks", tasks_str,
        "--num_fewshot", str(num_fewshot),
        "--limit", str(limit),
        "--batch_size", "auto",
        "--output_path", output_path,
    ]

    print(f"   Running lm-eval: {tasks_str}")
    print(f"   Model: {model_path}")
    print(f"   Limit: {limit} examples per task")

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=3600,
            env={**os.environ, "TF_CPP_MIN_LOG_LEVEL": "3"}
        )

        # Method 1: Find and parse results JSON file (PREFERRED)
        # Use recursive glob to find results file in any subfolder
        patterns = [
            f"{output_path}/**/results_*.json",
            f"{output_path}/**/results.json",
        ]

        results_file = None
        for pattern in patterns:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                results_file = sorted(matches)[-1]
                break

        if results_file and os.path.exists(results_file):
            print(f"   Found: {os.path.basename(results_file)}")
            with open(results_file) as f:
                full_results = json.load(f)

            # Extract the scores we need
            parsed = {}
            for task, data in full_results.get("results", {}).items():
                if isinstance(data, dict):
                    # GSM8K uses "exact_match,strict-match"
                    if "exact_match,strict-match" in data:
                        parsed[task] = data["exact_match,strict-match"]
                    # MATH uses "exact_match,none"
                    elif "exact_match,none" in data:
                        parsed[task] = data["exact_match,none"]

            print(f"   GSM8K: {parsed.get('gsm8k', 0)*100:.1f}%")
            print(f"   MATH: {parsed.get('hendrycks_math', 0)*100:.1f}%")

            return {"results": parsed, "source": "json_file"}

        # Method 2: Fallback to stdout parsing
        print(f"   No results.json found, parsing stdout...")
        parsed = parse_lm_eval_stdout(result.stdout)

        if parsed:
            return {"results": parsed, "source": "stdout_parsed"}

        return {"results": {}, "error": "no_results_found"}

    except subprocess.TimeoutExpired:
        return {"error": "timeout"}
    except Exception as e:
        return {"error": str(e)}


print("Fixed run_lm_eval_harness with recursive file search!")


def evaluate_math_model_benchmarks(model_config, limit: int = 10) -> Dict:
    """Evaluate a math model using lm-evaluation-harness."""
    print(f"\n{'='*60}")
    print(f"{model_config.name}")
    print(f"{'='*60}")

    output_path = f"{RESULTS_DIR}/math/{model_config.name.replace(' ', '_')}_lm_eval"

    results = run_lm_eval_harness(
        model_path=model_config.hf_path,
        tasks=["gsm8k", "hendrycks_math"],
        output_path=output_path,
        num_fewshot=5,
        limit=limit
    )

    return results


print("lm-evaluation-harness functions ready!")

Fixed run_lm_eval_harness with recursive file search!
lm-evaluation-harness functions ready!


In [22]:
# DS-1000 & HUMANEVAL SETUP FOR CODE MODELS (FIXED VERSION)

import subprocess
import os
import sys
import tempfile
import random
from datasets import load_dataset

print("Setting up code evaluation environment...")

# Install data science libraries for DS-1000 execution
!pip install -q numpy pandas scipy matplotlib scikit-learn

print("\n1. Setting up Direct HumanEval evaluation...")

def normalize_indentation(code: str) -> str:
    """
    Fix indentation by tracking indent level changes line-by-line.
    Uses the DIRECTION of indent changes rather than absolute values.
    """
    lines = code.split('\n')
    if not lines:
        return code

    # Get indent of each non-empty line
    line_data = []
    for line in lines:
        if line.strip():
            indent = len(line) - len(line.lstrip())
            line_data.append((indent, line.strip()))
        else:
            line_data.append((None, ''))

    if not line_data:
        return code

    # Build output by tracking indent level based on changes
    result = []
    current_level = 1  # Start at level 1 (inside function)
    prev_indent = None
    indent_stack = [0]  # Stack of indent values we've seen

    for indent, content in line_data:
        if indent is None:
            result.append('')
            continue

        if prev_indent is None:
            # First line
            result.append('    ' * current_level + content)
            indent_stack = [indent]
        else:
            if indent > prev_indent:
                # Indent increased - go deeper
                current_level += 1
                indent_stack.append(indent)
            elif indent < prev_indent:
                # Indent decreased - go back
                # Find how many levels we need to go back
                while indent_stack and indent_stack[-1] > indent:
                    indent_stack.pop()
                    current_level -= 1
                # Make sure we don't go below level 1
                current_level = max(1, current_level)
                if not indent_stack or indent_stack[-1] != indent:
                    indent_stack.append(indent)
            # else: same indent, same level

            result.append('    ' * current_level + content)

        prev_indent = indent

    return '\n'.join(result)


def run_humaneval_direct(model, tokenizer, model_name: str,
                         output_dir: str, limit: int = 10) -> Dict:
    """
    Run HumanEval evaluation with proper indentation normalization.
    """
    import tempfile

    os.makedirs(output_dir, exist_ok=True)

    print(f"   Running HumanEval (direct evaluation)...")
    print(f"   Model: {model_name}")
    print(f"   Limit: {limit} problems")

    # Load HumanEval
    try:
        humaneval = load_dataset("openai/openai_humaneval", split="test")
    except:
        humaneval = load_dataset("openai_humaneval", split="test")

    # Sample problems
    random.seed(42)
    if limit < len(humaneval):
        indices = random.sample(range(len(humaneval)), limit)
        problems = [humaneval[i] for i in indices]
    else:
        problems = list(humaneval)

    passed = 0
    total = len(problems)
    results_detail = []

    for problem in tqdm(problems, desc="HumanEval"):
        prompt = problem["prompt"]
        test = problem["test"]
        entry_point = problem["entry_point"]
        task_id = problem["task_id"]

        try:
            # Generate solution
            response = generate_response(model, tokenizer, prompt, max_new_tokens=512)

            # Clean up response
            clean_response = response.strip()

            # Remove markdown code blocks
            if "```python" in clean_response:
                clean_response = clean_response.split("```python")[1].split("```")[0]
            elif "```" in clean_response:
                parts = clean_response.split("```")
                if len(parts) >= 2:
                    clean_response = parts[1]

            clean_response = clean_response.strip()

            # Stop at new function/class definition (top-level, not indented)
            lines = clean_response.split('\n')
            extracted_lines = []
            for line in lines:
                # Stop if we hit a new top-level definition
                if line.startswith('def ') or line.startswith('class '):
                    break
                # Stop at doctest markers
                if line.strip().startswith('>>>'):
                    break
                extracted_lines.append(line)

            clean_response = '\n'.join(extracted_lines).rstrip()

            # Normalize indentation
            clean_response = normalize_indentation(clean_response)

            # Build full code
            full_code = prompt + clean_response + "\n\n" + test + f"\ncheck({entry_point})"

            # Execute
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(full_code)
                temp_file = f.name

            try:
                result = subprocess.run(
                    ['python', temp_file],
                    capture_output=True,
                    text=True,
                    timeout=15
                )
                is_passed = result.returncode == 0
                error_msg = result.stderr[:300] if result.stderr else ""
            except subprocess.TimeoutExpired:
                is_passed = False
                error_msg = "Timeout"
            finally:
                try:
                    os.unlink(temp_file)
                except:
                    pass

            if is_passed:
                passed += 1

            results_detail.append({
                "task_id": task_id,
                "passed": is_passed,
                "error": error_msg if not is_passed else "",
            })

        except Exception as e:
            results_detail.append({
                "task_id": task_id,
                "passed": False,
                "error": str(e)[:200]
            })

    pass_at_1 = passed / total if total > 0 else 0
    print(f"   HumanEval Pass@1: {pass_at_1*100:.1f}% ({passed}/{total})")

    summary = {
        "humaneval": {
            "pass@1": pass_at_1,
            "passed": passed,
            "total": total
        },
        "details": results_detail
    }

    with open(f"{output_dir}/humaneval_results.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary


def run_ds1000_evaluation_lenient(model, tokenizer, model_name: str,
                                  output_dir: str, limit: int = 10) -> Dict:
    """
    Run DS-1000 evaluation - handles natural language prompts.
    """
    import numpy as np
    import pandas as pd

    os.makedirs(output_dir, exist_ok=True)

    print(f"   Running DS-1000 evaluation...")
    print(f"   Model: {model_name}")
    print(f"   Limit: {limit} problems")

    # Load DS-1000
    try:
        ds1000 = load_dataset("xlangai/DS-1000", split="test")
        ds1000 = list(ds1000)
        print(f"   Loaded {len(ds1000)} DS-1000 problems")
    except Exception as e:
        print(f"   Failed to load DS-1000: {e}")
        return {"error": str(e)}

    # Sample problems
    random.seed(42)
    if limit and limit < len(ds1000):
        problems = random.sample(ds1000, limit)
    else:
        problems = ds1000

    results_by_lib = {}
    all_results = []

    for i, problem in enumerate(tqdm(problems, desc="DS-1000")):
        lib = problem.get("metadata", {}).get("library", "unknown") if isinstance(problem.get("metadata"), dict) else "unknown"
        prompt_text = problem.get("prompt", "")

        # Extract any code from the prompt (setup code)
        setup_code_lines = []

        for line in prompt_text.split('\n'):
            stripped = line.strip()
            if stripped.startswith(('import ', 'from ', 'data =', 'df =', 'arr =', 'x =', 'y =')):
                setup_code_lines.append(line)
            elif stripped.startswith('>>>'):
                setup_code_lines.append(stripped[3:].strip())

        setup_code = '\n'.join(setup_code_lines)

        # Create prompt for the model
        generation_prompt = f"""{prompt_text}

Write Python code to solve this problem. Output only the code, no explanations.
```python
"""

        try:
            response = generate_response(model, tokenizer, generation_prompt, max_new_tokens=300)

            # Extract code from response
            code = response.strip()

            # Remove markdown if present
            if "```python" in code:
                code = code.split("```python")[0]
            if "```" in code:
                code = code.split("```")[0]

            code = code.strip()

            # Remove non-code prefix lines
            lines = code.split('\n')
            code_lines = []
            for line in lines:
                stripped = line.strip()
                if stripped.lower().startswith(('here', 'the solution', 'this code', 'output:', 'answer:')):
                    continue
                if not code_lines and not stripped:
                    continue
                code_lines.append(line)

            code = '\n'.join(code_lines).strip()

            # Build execution code
            exec_code = ""
            if setup_code:
                exec_code = setup_code + "\n\n"
            exec_code += code

            # Try to execute
            try:
                import scipy
                import scipy.stats

                exec_globals = {
                    '__builtins__': __builtins__,
                    'np': np,
                    'numpy': np,
                    'pd': pd,
                    'pandas': pd,
                    'DataFrame': pd.DataFrame,
                    'Series': pd.Series,
                    'scipy': scipy,
                }

                try:
                    import matplotlib
                    matplotlib.use('Agg')
                    import matplotlib.pyplot as plt
                    exec_globals['plt'] = plt
                    exec_globals['matplotlib'] = matplotlib
                except:
                    pass

                exec(compile(exec_code, '<string>', 'exec'), exec_globals)
                is_passed = True
                error_msg = ""

            except Exception as e:
                is_passed = False
                error_msg = str(e)[:150]

        except Exception as e:
            is_passed = False
            error_msg = str(e)[:150]

        result_entry = {
            "problem_id": i,
            "library": lib,
            "passed": is_passed,
            "error": error_msg,
        }
        all_results.append(result_entry)

        if lib not in results_by_lib:
            results_by_lib[lib] = {"total": 0, "passed": 0}
        results_by_lib[lib]["total"] += 1
        if is_passed:
            results_by_lib[lib]["passed"] += 1

    overall_passed = sum(r["passed"] for r in all_results)
    overall_total = len(all_results)

    summary = {
        "model": model_name,
        "overall_pass_rate": overall_passed / overall_total if overall_total > 0 else 0,
        "overall_passed": overall_passed,
        "overall_total": overall_total,
        "by_library": {}
    }

    for lib, counts in results_by_lib.items():
        summary["by_library"][lib] = {
            "pass_rate": counts["passed"] / counts["total"] if counts["total"] > 0 else 0,
            "passed": counts["passed"],
            "total": counts["total"]
        }

    with open(f"{output_dir}/ds1000_results.json", "w") as f:
        json.dump({"summary": summary, "detailed": all_results}, f, indent=2)

    print(f"   DS-1000 Pass@1: {summary['overall_pass_rate']*100:.1f}%")

    return summary

Setting up code evaluation environment...

1. Setting up Direct HumanEval evaluation...


In [9]:
# CUSTOM EVALUATION ON OUR TEST SPLITS

def clear_gpu():
    """Clear GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    gc.collect()

def load_model_4bit(model_config: ModelConfig):
    """Load model in 4-bit for inference."""
    clear_gpu()

    model_path = model_config.unsloth_path or model_config.hf_path

    print(f"   Loading: {model_path}")

    try:
        # Try Unsloth first
        if model_config.unsloth_path:
            from unsloth import FastLanguageModel
            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=model_config.unsloth_path,
                max_seq_length=model_config.max_seq_length,
                dtype=None,
                load_in_4bit=True,
            )
            FastLanguageModel.for_inference(model)
            return model, tokenizer, "unsloth"
    except Exception as e:
        print(f"   Unsloth failed, using standard: {e}")

    # Standard loading
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_config.hf_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_config.hf_path,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    return model, tokenizer, "standard"


def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    """Generate response from model."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()


def extract_messages(item: Dict) -> tuple:
    """Extract system, user, assistant messages from our JSONL format."""
    system, user, assistant = "", "", ""
    for msg in item.get("messages", []):
        if msg["role"] == "system":
            system = msg["content"]
        elif msg["role"] == "user":
            user = msg["content"]
        elif msg["role"] == "assistant":
            assistant = msg["content"]
    return system, user, assistant


def evaluate_on_custom_test(model, tokenizer, test_data: List[Dict],
                            category: str, model_name: str,
                            max_samples: int = 30) -> Dict:
    """
    Evaluate model on our custom test split using LLM-as-Judge.
    """
    results = {
        "model": model_name,
        "category": category,
        "num_samples": min(max_samples, len(test_data)),
        "scores": [],
        "avg_overall": 0,
        "responses": []
    }

    samples = test_data[:max_samples]

    for item in tqdm(samples, desc=f"Evaluating {model_name}"):
        system, user, reference = extract_messages(item)

        # Generate response
        prompt = f"{system}\n\nQuestion: {user}\n\nAnswer:"
        response = generate_response(model, tokenizer, prompt)

        # Judge the response
        judge_result = llm_judge_score(user, reference, response, category)

        results["scores"].append(judge_result.get("overall", 0))
        results["responses"].append({
            "question": user[:200],
            "reference": reference[:200],
            "response": response[:200],
            "score": judge_result.get("overall", 0),
            "feedback": judge_result.get("brief_feedback", "")
        })

    results["avg_overall"] = np.mean(results["scores"]) if results["scores"] else 0

    return results


print("Custom evaluation functions defined!")

Custom evaluation functions defined!


In [10]:
# =============================================================================
# EVALUATE ALL MATH MODELS (OPTIMIZED - Benchmarks + LLM-Judge with 10 samples)
# =============================================================================

MATH_RESULTS = {}
BENCHMARK_LIMIT = 10      # For GSM8K, MATH
LLM_JUDGE_SAMPLES = 10    # Reduced from 30 for speed

print("="*80)
print("EVALUATING MATH MODELS (5 models)")
print("="*80)
print(f"   Benchmark limit: {BENCHMARK_LIMIT} samples per task")
print(f"   LLM-Judge samples: {LLM_JUDGE_SAMPLES}")
print(f"   Estimated time: ~15 mins per model")
print("="*80)

for i, model_config in enumerate(MATH_MODELS):
    start_time = time.time()

    print(f"\n{'='*80}")
    print(f"[{i+1}/5] {model_config.name}")
    print("="*80)

    model_results = {
        "name": model_config.name,
        "hf_path": model_config.hf_path,
        "unsloth_path": model_config.unsloth_path,
        "benchmarks": {},
        "custom_test": {},
    }

    # ----- 1. Run lm-eval benchmarks -----
    print("\nRunning lm-evaluation-harness (GSM8K, MATH)...")
    try:
        bench_results = evaluate_math_model_benchmarks(model_config, limit=BENCHMARK_LIMIT)
        model_results["benchmarks"] = bench_results

        # Extract and print scores
        parsed = bench_results.get("results", {})
        if parsed:
            gsm8k = parsed.get("gsm8k", "N/A")
            math_score = parsed.get("hendrycks_math", "N/A")
            if isinstance(gsm8k, float):
                print(f"   GSM8K: {gsm8k*100:.1f}%")
            if isinstance(math_score, float):
                print(f"   MATH: {math_score*100:.1f}%")

    except Exception as e:
        print(f"   Benchmark failed: {e}")
        model_results["benchmarks"] = {"error": str(e)}

    # ----- 2. Run LLM-as-Judge on custom test -----
    print(f"\nRunning LLM-as-Judge ({LLM_JUDGE_SAMPLES} samples)...")
    try:
        model, tokenizer, loader = load_model_4bit(model_config)

        custom_results = evaluate_on_custom_test(
            model, tokenizer, math_test, "math", model_config.name,
            max_samples=LLM_JUDGE_SAMPLES  # Use reduced sample size
        )
        model_results["custom_test"] = custom_results
        print(f"   LLM-Judge Avg Score: {custom_results.get('avg_overall', 0):.2f}/5")

        # Unload model to free memory
        del model, tokenizer
        clear_gpu()

    except Exception as e:
        print(f"   LLM-Judge failed: {e}")
        model_results["custom_test"] = {"error": str(e), "avg_overall": 0}

    # ----- Calculate elapsed time -----
    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed/60:.1f} minutes")

    # ----- Save results -----
    MATH_RESULTS[model_config.name] = model_results

    results_path = f"{RESULTS_DIR}/math/{model_config.name.replace(' ', '_')}_results.json"
    with open(results_path, "w") as f:
        json.dump(model_results, f, indent=2, default=str)
    print(f"Saved: {results_path}")

    print(f"\n{model_config.name} complete!")

    # Cool down before next model
    clear_gpu()
    time.sleep(3)

print("\n" + "="*80)
print("ALL MATH MODELS EVALUATED!")
print("="*80)

# ----- Quick Summary Table -----
print("\nMATH MODELS SUMMARY:")
print("-" * 70)
print(f"{'Model':<35} {'GSM8K':<10} {'MATH':<10} {'LLM-Judge':<10}")
print("-" * 70)

for name, data in MATH_RESULTS.items():
    # Extract benchmark scores
    bench = data.get("benchmarks", {}).get("results", {})
    gsm8k = bench.get("gsm8k", 0)
    math_score = bench.get("hendrycks_math", 0)

    # Extract LLM-Judge score
    judge = data.get("custom_test", {}).get("avg_overall", 0)

    # Format scores
    gsm8k_str = f"{gsm8k*100:.1f}%" if isinstance(gsm8k, float) else "N/A"
    math_str = f"{math_score*100:.1f}%" if isinstance(math_score, float) else "N/A"
    judge_str = f"{judge:.2f}/5" if isinstance(judge, (int, float)) else "N/A"

    print(f"{name:<35} {gsm8k_str:<10} {math_str:<10} {judge_str:<10}")

print("-" * 70)

EVALUATING MATH MODELS (5 models)
   Benchmark limit: 10 samples per task
   LLM-Judge samples: 10
   Estimated time: ~15 mins per model

[1/5] DeepSeek-Math-7B-Instruct

Running lm-evaluation-harness (GSM8K, MATH)...

DeepSeek-Math-7B-Instruct
   Running lm-eval: gsm8k,hendrycks_math
   Model: deepseek-ai/deepseek-math-7b-instruct
   Limit: 10 examples per task
   Found: results_2025-12-02T11-49-34.167556.json
   GSM8K: 40.0%
   MATH: 17.1%
   GSM8K: 40.0%
   MATH: 17.1%

Running LLM-as-Judge (10 samples)...
   Loading: deepseek-ai/deepseek-math-7b-instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating DeepSeek-Math-7B-Instruct:   0%|          | 0/10 [00:00<?, ?it/s]

   LLM-Judge Avg Score: 2.73/5

Total time: 26.4 minutes
Saved: ./results/math/DeepSeek-Math-7B-Instruct_results.json

DeepSeek-Math-7B-Instruct complete!

[2/5] AceMath-7B-Instruct

Running lm-evaluation-harness (GSM8K, MATH)...

AceMath-7B-Instruct
   Running lm-eval: gsm8k,hendrycks_math
   Model: nvidia/AceMath-7B-Instruct
   Limit: 10 examples per task
   Found: results_2025-12-02T12-23-12.968189.json
   GSM8K: 60.0%
   MATH: 28.6%
   GSM8K: 60.0%
   MATH: 28.6%

Running LLM-as-Judge (10 samples)...
   Loading: nvidia/AceMath-7B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Evaluating Llemma-7B:   0%|          | 0/10 [00:00<?, ?it/s]

   LLM-Judge Avg Score: 1.62/5

Total time: 27.6 minutes
Saved: ./results/math/Llemma-7B_results.json

Llemma-7B complete!

[5/5] WizardMath-7B-V1.1

Running lm-evaluation-harness (GSM8K, MATH)...

WizardMath-7B-V1.1
   Running lm-eval: gsm8k,hendrycks_math
   Model: WizardLMTeam/WizardMath-7B-V1.1
   Limit: 10 examples per task
   Found: results_2025-12-02T14-09-13.968617.json
   GSM8K: 80.0%
   MATH: 7.1%
   GSM8K: 80.0%
   MATH: 7.1%

Running LLM-as-Judge (10 samples)...
   Loading: WizardLMTeam/WizardMath-7B-V1.1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating WizardMath-7B-V1.1:   0%|          | 0/10 [00:00<?, ?it/s]

   LLM-Judge Avg Score: 2.20/5

Total time: 44.3 minutes
Saved: ./results/math/WizardMath-7B-V1.1_results.json

WizardMath-7B-V1.1 complete!

ALL MATH MODELS EVALUATED!

MATH MODELS SUMMARY:
----------------------------------------------------------------------
Model                               GSM8K      MATH       LLM-Judge 
----------------------------------------------------------------------
DeepSeek-Math-7B-Instruct           40.0%      17.1%      2.73/5    
AceMath-7B-Instruct                 60.0%      28.6%      3.25/5    
Qwen2.5-Math-7B-Instruct            90.0%      25.7%      3.25/5    
Llemma-7B                           30.0%      10.0%      1.62/5    
WizardMath-7B-V1.1                  80.0%      7.1%       2.20/5    
----------------------------------------------------------------------


In [23]:
# =============================================================================
# EVALUATE ALL CODE MODELS (Direct HumanEval + DS-1000 Lenient + LLM-as-Judge)
# =============================================================================

CODE_RESULTS = {}

# Evaluation limits
HUMANEVAL_LIMIT = 10
DS1000_LIMIT = 10
CUSTOM_TEST_LIMIT = 10

print("="*80)
print("EVALUATING CODE MODELS (5 models)")
print("="*80)
print(f"   HumanEval: {HUMANEVAL_LIMIT} problems (direct evaluation)")
print(f"   DS-1000: {DS1000_LIMIT} problems (lenient mode)")
print(f"   LLM-Judge: {CUSTOM_TEST_LIMIT} examples (strict GPT-4o)")
print("="*80)

for i, model_config in enumerate(CODING_MODELS):
    start_time = time.time()

    print(f"\n{'='*80}")
    print(f"[{i+1}/5] {model_config.name}")
    print("="*80)

    model_results = {
        "name": model_config.name,
        "hf_path": model_config.hf_path,
        "unsloth_path": model_config.unsloth_path,
        "humaneval": {},
        "ds1000": {},
        "custom_test": {},
    }

    # Load model
    try:
        print(f"\nLoading model...")
        model, tokenizer, loader_type = load_model_4bit(model_config)
        model_loaded = True
        print(f"   Loaded with: {loader_type}")
    except Exception as e:
        print(f"   Failed to load model: {e}")
        model_loaded = False

    # ----- 1. HumanEval (Direct) -----
    print(f"\n{'─'*60}")
    print(f"[1/3] HumanEval Benchmark (Direct Evaluation)")
    print(f"{'─'*60}")

    if model_loaded:
        try:
            humaneval_results = run_humaneval_direct(
                model=model,
                tokenizer=tokenizer,
                model_name=model_config.name,
                output_dir=f"{RESULTS_DIR}/code/{model_config.name.replace(' ', '_')}/humaneval",
                limit=HUMANEVAL_LIMIT
            )
            model_results["humaneval"] = humaneval_results
        except Exception as e:
            print(f"   HumanEval failed: {e}")
            model_results["humaneval"] = {"error": str(e)}
    else:
        model_results["humaneval"] = {"error": "Model not loaded"}

    # ----- 2. DS-1000 (Lenient) -----
    print(f"\n{'─'*60}")
    print(f"[2/3] DS-1000 Benchmark (Lenient Mode)")
    print(f"{'─'*60}")

    if model_loaded:
        try:
            ds1000_results = run_ds1000_evaluation_lenient(
                model=model,
                tokenizer=tokenizer,
                model_name=model_config.name,
                output_dir=f"{RESULTS_DIR}/code/{model_config.name.replace(' ', '_')}/ds1000",
                limit=DS1000_LIMIT
            )
            model_results["ds1000"] = ds1000_results
        except Exception as e:
            print(f"   DS-1000 failed: {e}")
            model_results["ds1000"] = {"error": str(e)}
    else:
        model_results["ds1000"] = {"error": "Model not loaded"}

    # ----- 3. Custom Test with LLM-as-Judge -----
    print(f"\n{'─'*60}")
    print(f"[3/3] LLM-as-Judge (Strict GPT-4o)")
    print(f"{'─'*60}")

    if model_loaded:
        try:
            custom_results = evaluate_on_custom_test(
                model, tokenizer, code_test, "code",
                model_config.name, max_samples=CUSTOM_TEST_LIMIT
            )
            model_results["custom_test"] = custom_results
            print(f"   LLM-Judge Avg Score: {custom_results['avg_overall']:.2f}/5")
        except Exception as e:
            print(f"   LLM-Judge failed: {e}")
            model_results["custom_test"] = {"error": str(e), "avg_overall": 0}
    else:
        model_results["custom_test"] = {"error": "Model not loaded", "avg_overall": 0}

    # Unload model
    if model_loaded:
        del model, tokenizer
        clear_gpu()

    # Calculate elapsed time
    elapsed = time.time() - start_time
    print(f"\nTotal time: {elapsed/60:.1f} minutes")

    # Save results
    CODE_RESULTS[model_config.name] = model_results

    results_file = f"{RESULTS_DIR}/code/{model_config.name.replace(' ', '_')}_results.json"
    os.makedirs(os.path.dirname(results_file), exist_ok=True)
    with open(results_file, "w") as f:
        json.dump(model_results, f, indent=2, default=str)

    print(f"\n{model_config.name} complete!")
    print(f"   Results saved to: {results_file}")

    # Cool down
    clear_gpu()
    time.sleep(5)

print("\n" + "="*80)
print("ALL CODE MODELS EVALUATED!")
print("="*80)

# Quick summary
print("\nCODE MODELS SUMMARY:")
print("─" * 70)
print(f"{'Model':<30} {'HumanEval':<12} {'DS-1000':<12} {'LLM-Judge':<12}")
print("─" * 70)

for name, results in CODE_RESULTS.items():
    humaneval = results.get("humaneval", {}).get("humaneval", {}).get("pass@1", 0)
    ds1000 = results.get("ds1000", {}).get("overall_pass_rate", 0)
    judge = results.get("custom_test", {}).get("avg_overall", 0)

    h_str = f"{humaneval*100:.1f}%" if isinstance(humaneval, float) else "N/A"
    d_str = f"{ds1000*100:.1f}%" if isinstance(ds1000, float) else "N/A"
    j_str = f"{judge:.2f}/5" if isinstance(judge, (int, float)) else "N/A"

    print(f"{name:<30} {h_str:<12} {d_str:<12} {j_str:<12}")

print("─" * 70)

EVALUATING CODE MODELS (5 models)
   HumanEval: 10 problems (direct evaluation)
   DS-1000: 10 problems (lenient mode)
   LLM-Judge: 10 examples (strict GPT-4o)

[1/5] CodeGemma-7B-IT

Loading model...
   Loading: unsloth/codegemma-7b-it-bnb-4bit
==((====))==  Unsloth 2025.11.6: Fast Gemma patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
   Loaded with: unsloth

────────────────────────────────────────────────────────────
[1/3] HumanEval Benchmark (Direct Evaluation)
────────────────────────────────────────────────────────────
   Running HumanEval (direct evaluation)...
   Model: CodeGemma-7B-IT
   Limit: 10 problems

HumanEval:   0%|          | 0/10 [00:00<?, ?it/s]

   HumanEval Pass@1: 20.0% (2/10)

────────────────────────────────────────────────────────────
[2/3] DS-1000 Benchmark (Lenient Mode)
────────────────────────────────────────────────────────────
   Running DS-1000 evaluation...
   Model: CodeGemma-7B-IT
   Limit: 10 problems
   Loaded 1000 DS-1000 problems


DS-1000:   0%|          | 0/10 [00:00<?, ?it/s]

[ True  True  True  True  True  True  True]
   DS-1000 Pass@1: 10.0%

────────────────────────────────────────────────────────────
[3/3] LLM-as-Judge (Strict GPT-4o)
────────────────────────────────────────────────────────────


...

   HumanEval Pass@1: 40.0% (4/10)

────────────────────────────────────────────────────────────
[2/3] DS-1000 Benchmark (Lenient Mode)
────────────────────────────────────────────────────────────
   Running DS-1000 evaluation...
   Model: Seed-Coder-8B-Instruct
   Limit: 10 problems
   Loaded 1000 DS-1000 problems


DS-1000:   0%|          | 0/10 [00:00<?, ?it/s]

True
   DS-1000 Pass@1: 50.0%

────────────────────────────────────────────────────────────
[3/3] LLM-as-Judge (Strict GPT-4o)
────────────────────────────────────────────────────────────


Evaluating Seed-Coder-8B-Instruct:   0%|          | 0/10 [00:00<?, ?it/s]

   LLM-Judge Avg Score: 3.02/5

Total time: 5.9 minutes

Seed-Coder-8B-Instruct complete!
   Results saved to: ./results/code/Seed-Coder-8B-Instruct_results.json

ALL CODE MODELS EVALUATED!

CODE MODELS SUMMARY:
──────────────────────────────────────────────────────────────────────
Model                          HumanEval    DS-1000      LLM-Judge   
──────────────────────────────────────────────────────────────────────
CodeGemma-7B-IT                20.0%        10.0%        2.80/5      
Qwen2.5-Coder-7B-Instruct      30.0%        30.0%        2.95/5      
DeepSeek-Coder-6.7B-Instruct   50.0%        0.0%         2.90/5      
CodeLlama-7B-Instruct          30.0%        40.0%        1.82/5      
Seed-Coder-8B-Instruct         40.0%        50.0%        3.02/5      
──────────────────────────────────────────────────────────────────────


In [24]:
# EVALUATE ALL THEORY MODELS

THEORY_RESULTS = {}

print("="*80)
print("EVALUATING THEORY MODELS (5 models)")
print("="*80)

for i, model_config in enumerate(THEORY_MODELS):
    print(f"\n{'='*80}")
    print(f"[{i+1}/5] {model_config.name}")
    print("="*80)

    model_results = {
        "name": model_config.name,
        "hf_path": model_config.hf_path,
        "unsloth_path": model_config.unsloth_path,
        "custom_test": {},
    }

    # LLM-as-Judge evaluation
    print("\nRunning LLM-as-Judge on custom test split...")
    try:
        model, tokenizer, loader = load_model_4bit(model_config)
        custom_results = evaluate_on_custom_test(
            model, tokenizer, theory_test, "theory", model_config.name, max_samples=10
        )
        model_results["custom_test"] = custom_results
        print(f"   Custom test complete: Avg Score = {custom_results['avg_overall']:.2f}/5")

        del model, tokenizer
        clear_gpu()

    except Exception as e:
        print(f"   Custom test failed: {e}")
        model_results["custom_test"] = {"error": str(e)}

    # Save results
    THEORY_RESULTS[model_config.name] = model_results

    with open(f"{RESULTS_DIR}/theory/{model_config.name.replace(' ', '_')}_results.json", "w") as f:
        json.dump(model_results, f, indent=2, default=str)

    print(f"\n{model_config.name} evaluation complete!")
    clear_gpu()
    time.sleep(5)

print("\n" + "="*80)
print("ALL THEORY MODELS EVALUATED!")
print("="*80)

EVALUATING THEORY MODELS (5 models)

[1/5] Llama-3.1-8B-Instruct

Running LLM-as-Judge on custom test split...
   Loading: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Evaluating Llama-3.1-8B-Instruct:   0%|          | 0/10 [00:00<?, ?it/s]

   Custom test complete: Avg Score = 3.20/5

Llama-3.1-8B-Instruct evaluation complete!

[2/5] Mistral-7B-Instruct-v0.3

Running LLM-as-Judge on custom test split...
   Loading: unsloth/Mistral-7B-Instruct-v0.3-bnb-4bit
==((====))==  Unsloth 2025.11.6: Fast Mistral patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Evaluating Mistral-7B-Instruct-v0.3:   0%|          | 0/10 [00:00<?, ?it/s]

   Custom test complete: Avg Score = 3.17/5

Mistral-7B-Instruct-v0.3 evaluation complete!

[3/5] Gemma-2-9B-Instruct

Running LLM-as-Judge on custom test split...
   Loading: unsloth/gemma-2-9b-it-bnb-4bit
Unsloth: If you want to finetune Gemma 2, install flash-attn to make it faster!
To install flash-attn, do the below:

pip install --no-deps --upgrade "flash-attn>=2.6.3"
==((====))==  Unsloth 2025.11.6: Fast Gemma2 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

   Custom test complete: Avg Score = 3.27/5

Qwen2.5-7B-Instruct evaluation complete!

[5/5] Falcon-H1-7B-Instruct

Running LLM-as-Judge on custom test split...
   Loading: unsloth/Falcon-H1-7B-Instruct
==((====))==  Unsloth 2025.11.6: Fast Falcon_H1 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating Falcon-H1-7B-Instruct:   0%|          | 0/10 [00:00<?, ?it/s]

   Custom test complete: Avg Score = 3.40/5

Falcon-H1-7B-Instruct evaluation complete!

ALL THEORY MODELS EVALUATED!


In [25]:
# =============================================================================
# AGGREGATE RESULTS & CREATE LEADERBOARDS
# =============================================================================

def extract_score(data: Dict, keys: List[str], default=0):
    """Safely extract nested score."""
    try:
        result = data
        for key in keys:
            if isinstance(result, dict):
                result = result.get(key, default)
            else:
                return default
        return result if result is not None else default
    except:
        return default


def extract_benchmark_score(model_results: Dict, task: str) -> float:
    """Extract score from model results (handles multiple formats)."""
    # Path 1: benchmarks.results.task.exact_match (from results.json)
    try:
        bench = model_results.get("benchmarks", {})
        results = bench.get("results", {})

        if task == "gsm8k":
            score = results.get("gsm8k", {})
            if isinstance(score, dict):
                return float(score.get("exact_match,strict-match", 0))
            elif isinstance(score, (int, float)):
                return float(score)
        elif task == "hendrycks_math":
            score = results.get("hendrycks_math", {})
            if isinstance(score, dict):
                return float(score.get("exact_match,none", 0))
            elif isinstance(score, (int, float)):
                return float(score)
    except:
        pass

    # Path 2: benchmarks.results.task (direct float)
    try:
        bench = model_results.get("benchmarks", {})
        results = bench.get("results", {})
        score = results.get(task, 0)
        if isinstance(score, (int, float)):
            return float(score)
    except:
        pass

    return 0.0


# ----- MATH LEADERBOARD -----
print("="*80)
print("MATH MODELS LEADERBOARD")
print("="*80)

math_leaderboard = []
for name, results in MATH_RESULTS.items():
    gsm8k_score = extract_benchmark_score(results, "gsm8k")
    math_score = extract_benchmark_score(results, "hendrycks_math")
    judge_score = results.get("custom_test", {}).get("avg_overall", 0)

    if not isinstance(judge_score, (int, float)):
        judge_score = 0

    row = {
        "Model": name,
        "GSM8K (Acc)": gsm8k_score * 100,
        "MATH (Acc)": math_score * 100,
        "LLM-Judge": judge_score,
        "Has Unsloth": "✓" if any(m.unsloth_path for m in MATH_MODELS if m.name == name) else "○"
    }
    # Composite: 30% GSM8K + 30% MATH + 40% LLM-Judge (scaled)
    row["Composite"] = (row["GSM8K (Acc)"] * 0.3) + (row["MATH (Acc)"] * 0.3) + (row["LLM-Judge"] * 20 * 0.4)
    math_leaderboard.append(row)

math_df = pd.DataFrame(math_leaderboard).sort_values("Composite", ascending=False)
math_df["Rank"] = range(1, len(math_df) + 1)
math_df = math_df[["Rank", "Model", "GSM8K (Acc)", "MATH (Acc)", "LLM-Judge", "Composite", "Has Unsloth"]]

print("\nMath Models Ranking (Select Top 2):")
display(math_df)

top_2_math = math_df.head(2)["Model"].tolist()
print(f"\nSelected Math Models: {top_2_math}")


# ----- CODE LEADERBOARD -----
print("\n" + "="*80)
print("CODE MODELS LEADERBOARD")
print("="*80)

code_leaderboard = []
for name, results in CODE_RESULTS.items():
    # Extract HumanEval pass@1
    humaneval_score = results.get("humaneval", {}).get("humaneval", {}).get("pass@1", 0)
    if not isinstance(humaneval_score, (int, float)):
        humaneval_score = 0

    # Extract DS-1000 pass rate
    ds1000_score = results.get("ds1000", {}).get("overall_pass_rate", 0)
    if not isinstance(ds1000_score, (int, float)):
        ds1000_score = 0

    # Extract LLM-Judge score
    judge_score = results.get("custom_test", {}).get("avg_overall", 0)
    if not isinstance(judge_score, (int, float)):
        judge_score = 0

    row = {
        "Model": name,
        "HumanEval (Pass@1)": humaneval_score * 100,
        "DS-1000 (Pass@1)": ds1000_score * 100,
        "LLM-Judge": judge_score,
        "Has Unsloth": "✓" if any(m.unsloth_path for m in CODING_MODELS if m.name == name) else "○"
    }
    # Composite: 40% HumanEval + 30% DS-1000 + 30% LLM-Judge (scaled)
    row["Composite"] = (row["HumanEval (Pass@1)"] * 0.4 +
                       row["DS-1000 (Pass@1)"] * 0.3 +
                       row["LLM-Judge"] * 20 * 0.3)
    code_leaderboard.append(row)

code_df = pd.DataFrame(code_leaderboard).sort_values("Composite", ascending=False)
code_df["Rank"] = range(1, len(code_df) + 1)
code_df = code_df[["Rank", "Model", "HumanEval (Pass@1)", "DS-1000 (Pass@1)", "LLM-Judge", "Composite", "Has Unsloth"]]

print("\nCode Models Ranking (Select Top 1):")
display(code_df)

top_1_code = code_df.head(1)["Model"].tolist()
print(f"\nSelected Code Model: {top_1_code}")


# ----- THEORY LEADERBOARD -----
print("\n" + "="*80)
print("THEORY MODELS LEADERBOARD")
print("="*80)

theory_leaderboard = []
for name, results in THEORY_RESULTS.items():
    judge_score = results.get("custom_test", {}).get("avg_overall", 0)
    if not isinstance(judge_score, (int, float)):
        judge_score = 0

    row = {
        "Model": name,
        "LLM-Judge": judge_score,
        "Has Unsloth": "✓" if any(m.unsloth_path for m in THEORY_MODELS if m.name == name) else "○"
    }
    row["Composite"] = judge_score
    theory_leaderboard.append(row)

theory_df = pd.DataFrame(theory_leaderboard).sort_values("Composite", ascending=False)
theory_df["Rank"] = range(1, len(theory_df) + 1)
theory_df = theory_df[["Rank", "Model", "LLM-Judge", "Composite", "Has Unsloth"]]

print("\nTheory Models Ranking (Select Top 2):")
display(theory_df)

top_2_theory = theory_df.head(2)["Model"].tolist()
print(f"\nSelected Theory Models: {top_2_theory}")


# ----- FINAL SUMMARY -----
print("\n" + "="*80)
print("FINAL MODEL SELECTION")
print("="*80)
print(f"\nMath Models (Top 2): {top_2_math}")
print(f"Code Model (Top 1): {top_1_code}")
print(f"Theory Models (Top 2): {top_2_theory}")
print(f"\n Total: 5 models for fine-tuning")

MATH MODELS LEADERBOARD

Math Models Ranking (Select Top 2):


,Rank,Model,GSM8K (Acc),MATH (Acc),LLM-Judge,Composite,Has Unsloth
2,1,Qwen2.5-Math-7B-Instruct,90.0,25.714286,3.250,60.714286,✓
1,2,AceMath-7B-Instruct,60.0,28.571429,3.250,52.571429,○
4,3,WizardMath-7B-V1.1,80.0,7.142857,2.200,43.742857,○
0,4,DeepSeek-Math-7B-Instruct,40.0,17.142857,2.725,38.942857,○
3,5,Llemma-7B,30.0,10.000000,1.625,25.000000,○



Selected Math Models: ['Qwen2.5-Math-7B-Instruct', 'AceMath-7B-Instruct']

CODE MODELS LEADERBOARD

Code Models Ranking (Select Top 1):


,Rank,Model,HumanEval (Pass@1),DS-1000 (Pass@1),LLM-Judge,Composite,Has Unsloth
4,1,Seed-Coder-8B-Instruct,40.0,50.0,3.025,49.15,✓
1,2,Qwen2.5-Coder-7B-Instruct,30.0,30.0,2.950,38.70,✓
2,3,DeepSeek-Coder-6.7B-Instruct,50.0,0.0,2.900,37.40,○
3,4,CodeLlama-7B-Instruct,30.0,40.0,1.825,34.95,✓
0,5,CodeGemma-7B-IT,20.0,10.0,2.800,27.80,✓



Selected Code Model: ['Seed-Coder-8B-Instruct']

THEORY MODELS LEADERBOARD

Theory Models Ranking (Select Top 2):


,Rank,Model,LLM-Judge,Composite,Has Unsloth
4,1,Falcon-H1-7B-Instruct,3.400,3.400,✓
3,2,Qwen2.5-7B-Instruct,3.275,3.275,✓
2,3,Gemma-2-9B-Instruct,3.250,3.250,✓
0,4,Llama-3.1-8B-Instruct,3.200,3.200,✓
1,5,Mistral-7B-Instruct-v0.3,3.175,3.175,✓



Selected Theory Models: ['Falcon-H1-7B-Instruct', 'Qwen2.5-7B-Instruct']

FINAL MODEL SELECTION

Math Models (Top 2): ['Qwen2.5-Math-7B-Instruct', 'AceMath-7B-Instruct']
Code Model (Top 1): ['Seed-Coder-8B-Instruct']
Theory Models (Top 2): ['Falcon-H1-7B-Instruct', 'Qwen2.5-7B-Instruct']

 Total: 5 models for fine-tuning


In [28]:
# =============================================================================
# VISUALIZATIONS (FIXED)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.style.use('seaborn-v0_8-whitegrid')

# ----- 1. Math Models Comparison -----
ax1 = axes[0, 0]
# Check what columns exist
print("Math DF columns:", math_df.columns.tolist())

math_viz_cols = []
if "GSM8K (Acc)" in math_df.columns:
    math_viz_cols.append("GSM8K (Acc)")
if "MATH (Acc)" in math_df.columns:
    math_viz_cols.append("MATH (Acc)")
if "LLM-Judge" in math_df.columns:
    math_viz_cols.append("LLM-Judge")

if math_viz_cols:
    math_viz = math_df.set_index("Model")[math_viz_cols].copy()
    if "LLM-Judge" in math_viz.columns:
        math_viz["LLM-Judge"] = math_viz["LLM-Judge"] * 20  # Scale to 100
    math_viz.plot(kind="barh", ax=ax1, color=["#3498db", "#2ecc71", "#9b59b6"][:len(math_viz_cols)])
ax1.set_xlabel("Score")
ax1.set_title("Math Models Performance", fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')

# ----- 2. Code Models Comparison -----
ax2 = axes[0, 1]
print("Code DF columns:", code_df.columns.tolist())

code_viz_cols = []
if "HumanEval (Pass@1)" in code_df.columns:
    code_viz_cols.append("HumanEval (Pass@1)")
if "DS-1000 (Pass@1)" in code_df.columns:
    code_viz_cols.append("DS-1000 (Pass@1)")
if "LLM-Judge" in code_df.columns:
    code_viz_cols.append("LLM-Judge")

if code_viz_cols:
    code_viz = code_df.set_index("Model")[code_viz_cols].copy()
    if "LLM-Judge" in code_viz.columns:
        code_viz["LLM-Judge"] = code_viz["LLM-Judge"] * 20  # Scale to 100
    code_viz.plot(kind="barh", ax=ax2, color=["#e74c3c", "#f39c12", "#9b59b6"][:len(code_viz_cols)])
ax2.set_xlabel("Score")
ax2.set_title("Code Models Performance", fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')

# ----- 3. Theory Models Comparison -----
ax3 = axes[1, 0]
print("Theory DF columns:", theory_df.columns.tolist())

if "LLM-Judge" in theory_df.columns:
    theory_viz = theory_df.set_index("Model")[["LLM-Judge"]]
    colors = ['gold' if m in top_2_theory else '#3498db' for m in theory_viz.index]
    theory_viz.plot(kind="barh", ax=ax3, color='#3498db', legend=False)
ax3.set_xlabel("LLM-Judge Score (1-5)")
ax3.set_title("Theory Models Performance", fontsize=14, fontweight='bold')

# ----- 4. Final Selection Summary -----
ax4 = axes[1, 1]
ax4.axis('off')

# Handle cases where variables might not exist
top_2_theory_str = top_2_theory if 'top_2_theory' in dir() else ['TBD', 'TBD']
top_2_math_str = top_2_math if 'top_2_math' in dir() else ['TBD', 'TBD']
top_1_code_str = top_1_code if 'top_1_code' in dir() else ['TBD']

selection_text = f"""
FINAL MODEL SELECTION

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

THEORY (Top 2):
   1. {top_2_theory_str[0] if len(top_2_theory_str) > 0 else 'TBD'}
   2. {top_2_theory_str[1] if len(top_2_theory_str) > 1 else 'TBD'}

MATH (Top 2):
   1. {top_2_math_str[0] if len(top_2_math_str) > 0 else 'TBD'}
   2. {top_2_math_str[1] if len(top_2_math_str) > 1 else 'TBD'}

CODE (Top 1):
   1. {top_1_code_str[0] if len(top_1_code_str) > 0 else 'TBD'}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Total Models for Fine-tuning: 5
"""
ax4.text(0.1, 0.5, selection_text, fontsize=14, fontfamily='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/model_evaluation_summary.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nVisualization saved to {RESULTS_DIR}/model_evaluation_summary.png")

Math DF columns: ['Rank', 'Model', 'GSM8K (Acc)', 'MATH (Acc)', 'LLM-Judge', 'Composite', 'Has Unsloth']
Code DF columns: ['Rank', 'Model', 'HumanEval (Pass@1)', 'DS-1000 (Pass@1)', 'LLM-Judge', 'Composite', 'Has Unsloth']
Theory DF columns: ['Rank', 'Model', 'LLM-Judge', 'Composite', 'Has Unsloth']

Visualization saved to ./results/model_evaluation_summary.png


In [29]:
# SAVE FINAL RESULTS

final_results = {
    "evaluation_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "random_seed": RANDOM_SEED,
    "sample_sizes": {
        "custom_test": SAMPLE_SIZE,
        "gsm8k": GSM8K_SAMPLE_SIZE if 'GSM8K_SAMPLE_SIZE' in dir() else "N/A",
        "math": MATH_SAMPLE_SIZE if 'MATH_SAMPLE_SIZE' in dir() else "N/A",
        "humaneval": HUMANEVAL_SAMPLE_SIZE if 'HUMANEVAL_SAMPLE_SIZE' in dir() else "N/A",
    },
    "selected_models": {
        "theory": top_2_theory,
        "math": top_2_math,
        "code": top_1_code,
    },
    "leaderboards": {
        "math": math_df.to_dict(orient='records'),
        "code": code_df.to_dict(orient='records'),
        "theory": theory_df.to_dict(orient='records'),
    },
    "detailed_results": {
        "math": MATH_RESULTS,
        "code": CODE_RESULTS,
        "theory": THEORY_RESULTS,
    }
}

# Save locally
with open(f"{RESULTS_DIR}/final_evaluation_results.json", "w") as f:
    json.dump(final_results, f, indent=2, default=str)

print("Results saved locally!")

# Upload to GCS
print("\nUploading results to GCS...")

def upload_file_to_gcs(local_path, gcs_path):
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    print(f"   Uploaded: gs://{PROCESSED_BUCKET}/{gcs_path}")

upload_file_to_gcs(f"{RESULTS_DIR}/final_evaluation_results.json", "evaluation/final_evaluation_results.json")
upload_file_to_gcs(f"{RESULTS_DIR}/model_evaluation_summary.png", "evaluation/model_evaluation_summary.png")

print("\nAll results uploaded to GCS!")

Results saved locally!

Uploading results to GCS...
   Uploaded: gs://msda-llm-processed-data/evaluation/final_evaluation_results.json
   Uploaded: gs://msda-llm-processed-data/evaluation/model_evaluation_summary.png

All results uploaded to GCS!
